# 02 · Auditoría de conteos — Muerte fetal, MIMIC-IV (M3)

| Campo | Contenido |
|---|---|
| Secuencia del framework | 01: E1 · E2 · E3 · P1 → **02: E4** → 03: E5 · E6 → R3 → 04: R3 · E2 · E3 · P1 · E4 → 05: E5 · E6 · E7 · P2 → 06: P3 · clase · E11 |
| Qué hace | Corre la auditoría de conteos de los grupos A a D (base, eventos y definición, solapamiento de pacientes, era y codificación) y muestra las decisiones que cada grupo resolvió antes de cualquier modelo |
| Etapas y puertas | E4 (resuelve D1, D3, D4 y D5; prepara E5, E6 y el criterio C4) |
| Decisiones | D1 (base de datos), D3 (población), D4 (unidad), D5 (desenlace); ver `count_audit_es.md`, sección 4 |
| Entradas | Cohorte obstétrica revisada (`cohort_obstetric_v2`, `hadm_delivery_v2`, `obstetric_dx_v2`, `predictors_24h_v2`) |
| Script | `sql/02_audit_counts/audit_counts_es.sql` (solo lectura: tablas temporales y SELECT) |
| Salidas | `data/outputs_02_count_audit.txt` (solo conteos agregados; de 1 a 4 se muestran como "<5") |
| Tiempo | Uno a dos minutos |
| Anterior / siguiente | `01_outcome_definition` / `03_unit_and_prediction_point` |
| Versión en inglés | `02_count_audit.ipynb` (mismo código) |

In [1]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_02_count_audit.txt"
engine = get_connect_database()

## E4 · Auditoría de conteos, grupo A — conteos base y reconciliación (decisión D1)

La auditoría mide, antes de cualquier modelo, los números que deciden el diseño. Trabaja sobre la cohorte obstétrica revisada y sigue dos reglas: **sin solapamiento y sin selección por desenlace**. El grupo A comprueba que los conteos base coinciden con la auditoría materna y que la etiqueta reproduce el conjunto de datos anterior de muerte fetal (Q-A2 necesita la tabla anterior `cohort_fetal_death_dataset`; si no existe, solo esa consulta se reporta como fallida).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Los conteos base coinciden y la etiqueta reproduce el conjunto anterior | D1: usar las tablas revisadas de la cohorte obstétrica | 25.736 admisiones; 154 eventos en ambas; 0 discordantes |
| Difieren | Revisar la cohorte antes de seguir | — |

In [2]:
# Stage S4: count audit, groups A to D, in one connection (read-only script)
aud = run_script(engine, SQL / "02_audit_counts" / "audit_counts.sql", log=say)
show_results(aud, ["Q-A1 base", "Q-A2 reconciliation with the legacy dataset", "Q-A3 admissions per patient", "Q-A3b summary"])
a1 = aud["Q-A1 base"].iloc[0]
decision("S4 · Decision D1: database", [
    ("base counts", f"{int(a1.n_admissions):,} admissions; {int(a1.n_patients):,} patients; "
                    f"{int(a1.n_delivery_hospitalizations):,} delivery hospitalizations",
     "revised tables of the obstetric cohort (the same as the maternal count audit)")])

audit_counts.sql: 43 sentencias corridas (solo verificaciones), 0 omitidas, 0 errores, 85.1 s


,admisiones,pacientes,hospitalizaciones de parto,pacientes con parto,partos elegibles a 24 h,filas de predictors_24h_v2
1,"25,736","15,084","19,425","13,729","19,383","25,736"


,en ambas,solo en la cohorte revisada,solo en el conjunto anterior,eventos anteriores,eventos revisados,eventos en ambas,evento antes y no ahora,"evento nuevo, no en el anterior"
1,"25,736",0,0,154,154,154,0,0


,admisiones por paciente,pacientes,admisiones
1,1,"8,291","8,291"
2,2,"4,418","8,836"
3,3,"1,558","4,674"
4,4,491,"1,964"
5,5 o más,326,"1,971"


,pacientes,pacientes con más de una,% con más de una,máximo de admisiones
1,"15,084","6,793",45,27


,revisión,resultado,decisión
1,conteos base,"25,736 admisiones; 15,084 pacientes; 19,425 hospitalizaciones de parto",tablas revisadas de la cohorte obstétrica (las mismas de la auditoría materna)


## E4 · Auditoría de conteos, grupo B — eventos y definición

Cuántos eventos hay según el alcance y el nivel, cómo concuerdan con los códigos de mortinato de la marca de parto, cuántos tienen semanas de gestación y en qué posición está el código. La salida termina con la tabla «Decisiones D3 y D5».

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| 15 códigos de mortinato quedan fuera de la marca de parto, todos con código de aborto | D3: población = marca de parto **o** código de mortinato | 19.440 admisiones; 142 eventos, 129 estrictos |
| Las semanas de gestación solo existen en CIE-10; el 19% de los estrictos con semanas tiene menos de 22 | D5: código estricto a cualquier edad gestacional; 22 y 28 semanas como sensibilidad (solo CIE-10) | Etiqueta principal: 129 eventos estrictos |

In [3]:
# Group B: events and definition
show_results(aud, ["Q-B1 events by tier and ICD version", "Q-B2 events by scope", "Q-B3 label vs has_stillbirth_code",
                   "Q-B3b stillbirth code without delivery flag", "Q-B4 gestational weeks of events",
                   "Q-B4b weeks by delivery flag and abortive code", "Q-B5 position of the fetal-death code",
                   "Q-B6 strict codes outside the universe", "Q-B7 overlap with SMM and maternal death",
                   "Q-B8 events with abortive or ectopic codes"])
decision("S4 · Decisions D3 and D5", [
    ("stillbirth codes without the delivery flag (Q-B3b)", "all carry an abortive, ectopic or molar code",
     "D3: population = delivery flag or stillbirth code (19,440 admissions)"),
    ("strict codes outside the obstetric universe (Q-B6)", "none", "the universe does not lose events"),
    ("gestational weeks (Q-B4)", "only in ICD-10 admissions; some strict events under 22 weeks",
     "D5: strict code at any gestational age; 22 and 28 weeks as sensitivity (ICD-10 only)"),
    ("other outcomes in the same admission (Q-B7)", "severe maternal morbidity in some events; no maternal death",
     "no change: they are not part of the label")])

,era CIE,admisiones,eventos estrictos o parciales,eventos estrictos,eventos solo parciales,pacientes con evento
1,ambas,<5,0,0,0,0
2,solo CIE-10,"11,671",86,72,14,83
3,solo CIE-9,"14,064",68,64,<5,65
4,,"25,736",154,136,18,148


,alcance,admisiones,eventos estrictos o parciales,eventos estrictos,eventos por 1.000 admisiones
1,Cualquier admisión obstétrica,"25,736",154,136,5.98
2,Hospitalización de parto (marca v2),"19,425",127,114,6.54
3,No es hospitalización de parto,"6,311",27,22,4.28
4,Con un código obstétrico estructurado (cualquier posición),"25,644",154,136,6.01
5,Sin código obstétrico estructurado,92,0,0,0


,evento estricto,código de mortinato,marca de parto v2,era CIE,admisiones
1,Sí,Sí,Sí,solo CIE-10,60
2,Sí,Sí,Sí,solo CIE-9,49
3,Sí,Sí,No,solo CIE-10,10
4,Sí,Sí,No,solo CIE-9,5
5,Sí,No,Sí,solo CIE-10,<5
6,Sí,No,Sí,solo CIE-9,<5
7,Sí,No,No,solo CIE-10,<5
8,Sí,No,No,solo CIE-9,6
9,No,No,Sí,solo CIE-10,9
10,No,No,Sí,solo CIE-9,<5


,código de aborto,admisiones,eventos estrictos
1,Sí,15,15


,era CIE,nivel,edad gestacional,eventos
1,solo CIE-10,solo parcial,1 menos de 22 semanas,<5
2,solo CIE-10,solo parcial,3 28 semanas o más,11
3,solo CIE-10,estricto,1 menos de 22 semanas,13
4,solo CIE-10,estricto,2 22-27 semanas,19
5,solo CIE-10,estricto,3 28 semanas o más,37
6,solo CIE-10,estricto,4 sin código Z3A,<5
7,solo CIE-9,solo parcial,4 sin código Z3A,<5
8,solo CIE-9,estricto,4 sin código Z3A,64


,edad gestacional,marca de parto v2,código de aborto,eventos estrictos o parciales,eventos estrictos
1,1 menos de 20 semanas,No,Sí,7,6
2,1 menos de 20 semanas,Sí,No,<5,<5
3,2 20-21 semanas,No,No,<5,0
4,2 20-21 semanas,Sí,No,5,<5
5,3 22-27 semanas,No,No,<5,<5
6,3 22-27 semanas,No,Sí,<5,<5
7,3 22-27 semanas,Sí,No,17,17
8,4 28-36 semanas,No,No,<5,0
9,4 28-36 semanas,No,Sí,<5,<5
10,4 28-36 semanas,Sí,No,21,19


,posición más baja,eventos,eventos sin otro código obstétrico
1,1-4,118,<5
2,5-10,30,0
3,over 10,6,0


Q-B6 Códigos estrictos fuera del universo: no se obtuvo (ver los errores del script)


,nivel,eventos,con morbilidad materna severa,muerte materna en la misma admisión,en ambas
1,todos los eventos,154,15,0,0
2,solo parcial,18,0,0,0
3,estricto,136,15,0,0


,código de aborto,marca de parto v2,eventos estrictos o parciales,eventos estrictos
1,No,No,7,<5
2,No,Sí,127,114
3,Sí,No,20,19


,revisión,resultado,decisión
1,códigos de mortinato sin marca de parto (Q-B3b),"todos tienen un código de aborto, ectópico o molar",D3: población = marca de parto o código de mortinato (19.440 admisiones)
2,códigos estrictos fuera del universo obstétrico (Q-B6),ninguno,el universo no pierde eventos
3,semanas de gestación (Q-B4),solo en admisiones CIE-10; algunos eventos estrictos con menos de 22 semanas,D5: código estricto a cualquier edad gestacional; 22 y 28 semanas como sensibilidad (solo CIE-10)
4,otros desenlaces en la misma admisión (Q-B7),morbilidad materna severa en algunos eventos; ninguna muerte materna,sin cambio: no son parte de la etiqueta


## E4 · Auditoría de conteos, grupo C — solapamiento de pacientes y mismo embarazo (decisión D4)

Cuántas admisiones tiene cada paciente, qué tan separadas están y cuántos eventos conserva cada unidad de análisis. Decide el registro y la agrupación de la validación.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| El 45% de las pacientes tiene más de una admisión y el 40% de los eventos está en una admisión posterior | D4: un registro por hospitalización de parto; toda partición agrupada por paciente | Quedarse con la primera admisión perdería el 40% de los eventos |
| Las cadenas de embarazo cambian los eventos en 5 como máximo | La cadena de embarazo de 90 días queda como sensibilidad | 150, 149 y 149 eventos con cadenas de 30, 90 y 180 días |

In [4]:
# Group C: patient overlap and same pregnancy
show_results(aud, ["Q-C1 event admissions per patient", "Q-C2 gap to the previous admission",
                   "Q-C3 gap to an earlier fetal-death admission", "Q-C3b event admissions by rank",
                   "Q-C4 events by unit of analysis"])
c4 = aud["Q-C4 events by unit of analysis"].set_index("unit")
first = c4.loc["First admission per patient", "n_units_with_event"]
alladm = c4.loc["Admission (any obstetric)", "n_units_with_event"]
decision("S4 · Decision D4: unit of analysis", [
    ("events kept by the first admission only", f"{int(first)} of {int(alladm)} ({100 * (1 - first / alladm):.0f}% lost)",
     "not used: one record per delivery hospitalization"),
    ("patients with more than one admission", "frequent (Q-A3b)", "every validation split is grouped by patient"),
    ("pregnancy chains of 30, 90 and 180 days", ", ".join(str(int(v)) for v in c4.filter(like="Pregnancy chain", axis=0)["n_units_with_event"]) + " events",
     "the 90-day chain is a sensitivity analysis")])

,admisiones con evento de la paciente,pacientes,admisiones de estas pacientes
1,1,143,343
2,2,<5,12
3,3,<5,5


,intervalo con la admisión anterior,admisiones,eventos
1,0 primera admisión de la paciente,"15,084",93
2,1 menos de 90 días,"3,649",21
3,2 90-179 días,595,<5
4,3 180-364 días,415,<5
5,4 365 días o más,"5,993",37


,intervalo con un evento anterior,admisiones,eventos,no eventos
1,0 sin admisión anterior con muerte fetal,"25,626",148,"25,478"
2,1 menos de 90 días,17,5,12
3,3 180-364 días,15,0,15
4,4 365 días o más,78,<5,77


,orden de la admisión,eventos,eventos estrictos
1,primera admisión,93,86
2,admisión posterior,61,50


,unidad,unidades,unidades con evento,unidades con evento estricto,pacientes con evento
1,Admisión (cualquier obstétrica),"25,736",154,136,148
2,Hospitalización de parto,"19,425",127,114,126
3,Primera admisión por paciente,"15,084",93,86,93
4,"Cadena de embarazo, intervalo de 30 días como máximo","23,257",150,135,148
5,"Cadena de embarazo, intervalo de 90 días como máximo","22,080",149,134,148
6,"Cadena de embarazo, intervalo de 180 días como máximo","21,485",149,134,148


,revisión,resultado,decisión
1,eventos que conserva solo la primera admisión,93 de 154 (40% perdido),no se usa: un registro por hospitalización de parto
2,pacientes con más de una admisión,frecuentes (Q-A3b),toda partición de la validación se agrupa por paciente
3,"cadenas de embarazo de 30, 90 y 180 días","150, 149, 149 eventos",la cadena de 90 días es un análisis de sensibilidad


## E4 · Auditoría de conteos, grupo D — era y codificación

La CIE-9 cubre cerca de 2008–2014 y la CIE-10 cerca de 2014–2021, así que la versión de codificación y el período calendario están casi confundidos. La celda calcula además la prueba de tendencia ji cuadrado de la tasa de eventos en partos entre los grupos de año ancla (Cochran-Armitage), que la auditoría dejó pendiente de reproducir en el código.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| La tasa sube con el período y la versión de codificación se mueve con él | Se registra antes de cualquier modelo: aplica el criterio C4 (estabilidad en el tiempo) y se espera deriva de la calibración | De 5,2 a 12,9 por 1.000 partos |
| Semanas de gestación solo en CIE-10 | Las definiciones de 22 y 28 semanas se aplican solo a admisiones CIE-10 | El 45,5% de los partos tiene semanas |

In [5]:
# Group D: era and coding
show_results(aud, ["Q-D1 events by ICD version", "Q-D1b events by anchor-year group", "Q-D1c approximate year by ICD version",
                   "Q-D2 gestational weeks by ICD version", "Q-D3 rates per 1,000 deliveries"])
# Chi-square test for trend (Cochran-Armitage) of the event rate in deliveries across the anchor-year groups (Q-D1b)
from scipy.stats import norm
d1b = aud["Q-D1b events by anchor-year group"].dropna(subset=["anchor_year_group"])
d1b = d1b[d1b["n_deliveries"] > 0].sort_values("anchor_year_group")
x = np.arange(len(d1b), dtype=float)
nd, ev = d1b["n_deliveries"].to_numpy(float), d1b["n_events_in_deliveries"].to_numpy(float)
p_bar = ev.sum() / nd.sum()
z_trend = (x * (ev - nd * p_bar)).sum() / np.sqrt(p_bar * (1 - p_bar) * ((nd * x ** 2).sum() - (nd * x).sum() ** 2 / nd.sum()))
p_trend = 2 * norm.sf(abs(z_trend))
show("S4 · Test for trend of the event rate in deliveries (Cochran-Armitage)", pd.DataFrame([{
    "groups": len(d1b), "first group, per 1,000": round(1000 * ev[0] / nd[0], 2),
    "last group, per 1,000": round(1000 * ev[-1] / nd[-1], 2), "z": round(z_trend, 2), "p": round(p_trend, 4)}]))
decision("S4 · Era and coding", [
    ("event rate by period", f"from {1000 * ev[0] / nd[0]:.1f} to {1000 * ev[-1] / nd[-1]:.1f} per 1,000 deliveries (p = {p_trend:.3f})",
     "criterion C4 applies; calibration drift is expected; reported as a limitation"),
    ("ICD version and period", "almost confounded (Q-D1c)", "no sensitivity analysis by version can separate them"),
    ("gestational weeks", "only in ICD-10 admissions (Q-D2)", "22 and 28 week definitions only in ICD-10")])

,era CIE,partos,eventos en partos,estrictos en partos,eventos fuera de partos,eventos por 1.000 partos
1,ambas,0,0,0,0,
2,solo CIE-10,"8,883",70,61,16,7.88
3,solo CIE-9,"10,542",57,53,11,5.41
4,,"19,425",127,114,27,6.54


,grupo de año ancla,partos,eventos en partos,eventos fuera de partos,eventos por 1.000 partos
1,2008 - 2010,"8,227",43,7,5.23
2,2011 - 2013,"4,338",29,<5,6.69
3,2014 - 2016,"3,577",23,9,6.43
4,2017 - 2019,"2,349",20,7,8.51
5,2020 - 2022,934,12,<5,12.85


,era CIE,admisiones,año p05,año mediano,año p95
1,ambas,<5,"2,014","2,014","2,014"
2,solo CIE-10,"11,671","2,014","2,017","2,021"
3,solo CIE-9,"14,064","2,008","2,011","2,014"


,era CIE,eventos,eventos con semanas,partos,partos con semanas
1,ambas,0,0,0,0
2,solo CIE-10,86,83,"8,883","8,844"
3,solo CIE-9,68,0,"10,542",0
4,,154,83,"19,425","8,844"


,definición,eventos,partos,tasa por 1.000 partos
1,"A. Código de mortinato de la marca de parto (Z37.1/.4/.7, V27.1/.4/.7)",109,"19,425",5.61
2,B. Etiqueta estricta de muerte fetal,114,"19,425",5.87
3,C. Etiqueta estricta o parcial de muerte fetal,127,"19,425",6.54
4,"D. Solo CIE-10, estricta, 28 semanas o más (mortinato tardío)",36,"8,883",4.05


,grupos,"primer grupo, por 1.000","último grupo, por 1.000",z,p
1,5,5.23,12.85,2.71,0.0067


,revisión,resultado,decisión
1,tasa de eventos por período,"de 5.2 a 12.8 por 1,000 partos (p = 0.007)",aplica el criterio C4; se espera deriva de la calibración; se reporta como limitación
2,versión de la CIE y período,casi confundidos (Q-D1c),ningún análisis de sensibilidad por versión puede separarlos
3,semanas de gestación,solo en admisiones CIE-10 (Q-D2),definiciones de 22 y 28 semanas solo en CIE-10


## Resumen del notebook

Indicadores principales de este notebook y el paso siguiente.

In [6]:
# Summary of notebook 02
notebook_summary([
    ("obstetric admissions / patients", lambda: f"{int(a1.n_admissions):,} / {int(a1.n_patients):,}"),
    ("delivery hospitalizations", lambda: f"{int(a1.n_delivery_hospitalizations):,}"),
    ("events kept by the first admission only", lambda: f"{int(first)} of {int(alladm)}"),
    ("event rate in deliveries, first and last period", lambda: f"{1000 * ev[0] / nd[0]:.2f} and {1000 * ev[-1] / nd[-1]:.2f} per 1,000"),
    ("test for trend, p", lambda: round(p_trend, 4)),
    ("decisions settled", "D1 database, D3 population, D4 unit, D5 outcome"),
    ("statements of the script that failed", lambda: ", ".join(e[0] for e in aud["errors"]) or "none"),
    ("next step", "unit of analysis (S5) and prediction point (S6), notebook 03"),
])
write_outputs(OUT_FILE, "Notebook 02 (S4), fetal death MIMIC-IV (M3); aggregate output only")

,valor
indicador,
admisiones obstétricas / pacientes,"25,736 / 15,084"
hospitalizaciones de parto,"19,425"
eventos que conserva solo la primera admisión,93 de 154
"tasa de eventos en partos, primer y último período","5.23 y 12.85 por 1,000"
"prueba de tendencia, p",0.0067
decisiones resueltas,"D1 base de datos, D3 población, D4 unidad, D5 desenlace"
sentencias del script que fallaron,ninguno
paso siguiente,"unidad de análisis (E5) y punto de predicción (E6), notebook 03"


salida agregada escrita: data/outputs_02_count_audit.txt
